# Сравнение близости двух предложений

Сравниваем предложения:

- `роботы проектируют дроны`
- `дроны проектируют роботы`

У них одинаковый набор слов, но разный порядок.

Считаем cosine similarity четырьмя способами:

1. `CountVectorizer`
2. `TF-IDF`
3. Toy Transformer с ручным расчетом `Q`, `K`, `V`
4. Реальный Transformer (`bert-base-multilingual-cased`)

In [7]:
# Если пакеты не установлены, раскомментируйте строку ниже.
# %pip install -q numpy pandas scikit-learn torch transformers

In [9]:
import math
import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, BertModel

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
torch.set_grad_enabled(False)

sentences = [
    "автомобиль сломался",
    "машина вышла из строя",
]

sentences

['автомобиль сломался', 'машина вышла из строя']

## 1. CountVectorizer и TF-IDF

In [10]:
def vectorizer_report(vectorizer, label):
    matrix = vectorizer.fit_transform(sentences)
    dense = matrix.toarray().astype(float)
    frame = pd.DataFrame(
        dense,
        index=["sent_1", "sent_2"],
        columns=vectorizer.get_feature_names_out(),
    )
    score = float(cosine_similarity(dense[0:1], dense[1:2])[0, 0])
    print(label)
    display(frame)
    print(f"cosine similarity = {score:.4f}\n")
    return dense, score

count_vectors, count_score = vectorizer_report(CountVectorizer(), "CountVectorizer")
tfidf_vectors, tfidf_score = vectorizer_report(TfidfVectorizer(), "TF-IDF")

CountVectorizer


,автомобиль,вышла,из,машина,сломался,строя
sent_1,1.0,0.0,0.0,0.0,1.0,0.0
sent_2,0.0,1.0,1.0,1.0,0.0,1.0


cosine similarity = 0.0000

TF-IDF


,автомобиль,вышла,из,машина,сломался,строя
sent_1,0.707107,0.0,0.0,0.0,0.707107,0.0
sent_2,0.000000,0.5,0.5,0.5,0.000000,0.5


cosine similarity = 0.0000



## 2. Transformer

Покажем `Q`, `K`, `V` из первого слоя self-attention.

Здесь порядок слов уже учитывается внутри входных представлений модели через positional embeddings.

Важно: токены будут подсловными (`WordPiece`), а `Q/K/V` имеют форму `(batch, heads, seq_len, head_dim)`.

In [12]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.eval()

MODEL_NAME

Loading weights: 100%|█████████████| 199/199 [00:00<00:00, 2079.63it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

In [13]:
def mean_pool_without_special_tokens(last_hidden_state, special_tokens_mask):
    keep_mask = (1 - special_tokens_mask).unsqueeze(-1).float()
    pooled = (last_hidden_state * keep_mask).sum(dim=1) / keep_mask.sum(dim=1).clamp(min=1.0)
    return pooled.squeeze(0)


def _proj_to_heads(linear_out, num_heads, head_size):
    # linear_out: (batch, seq_len, all_head_size)
    # → (batch, num_heads, seq_len, head_size)
    b, s, _ = linear_out.shape
    return linear_out.view(b, s, num_heads, head_size).transpose(1, 2)


def real_transformer_report(sentence):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        return_special_tokens_mask=True,
        truncation=True,
    )

    embedding_output = model.embeddings(
        input_ids=inputs["input_ids"],
        token_type_ids=inputs.get("token_type_ids"),
    )

    sa = model.encoder.layer[0].attention.self
    num_heads = sa.num_attention_heads
    head_size = sa.attention_head_size

    q = _proj_to_heads(sa.query(embedding_output), num_heads, head_size)
    k = _proj_to_heads(sa.key(embedding_output), num_heads, head_size)
    v = _proj_to_heads(sa.value(embedding_output), num_heads, head_size)

    scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(head_size)
    probs = torch.softmax(scores, dim=-1)

    outputs = model(
        input_ids=inputs["input_ids"],
        token_type_ids=inputs.get("token_type_ids"),
        attention_mask=inputs.get("attention_mask"),
    )

    sentence_vector = mean_pool_without_special_tokens(
        outputs.last_hidden_state,
        inputs["special_tokens_mask"],
    ).cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    return {
        "tokens": tokens,
        "q": q.cpu(),
        "k": k.cpu(),
        "v": v.cpu(),
        "probs": probs.cpu(),
        "sentence_vector": sentence_vector,
    }

In [14]:
real_reports = [real_transformer_report(sentence) for sentence in sentences]

for i, report in enumerate(real_reports, start=1):
    tokens = report["tokens"]
    print(f"sentence_{i}: {sentences[i - 1]}")
    print("tokens:", tokens)
    print("Q shape:", tuple(report["q"].shape))
    print("K shape:", tuple(report["k"].shape))
    print("V shape:", tuple(report["v"].shape))

    head0_q = pd.DataFrame(report["q"][0, 0].numpy(), index=tokens)
    head0_k = pd.DataFrame(report["k"][0, 0].numpy(), index=tokens)
    head0_v = pd.DataFrame(report["v"][0, 0].numpy(), index=tokens)
    head0_attn = pd.DataFrame(report["probs"][0, 0].numpy(), index=tokens, columns=tokens)

    print("head 0: Q")
    display(head0_q.round(3))
    print("head 0: K")
    display(head0_k.round(3))
    print("head 0: V")
    display(head0_v.round(3))
    print("head 0: attention weights A = softmax(QK^T / sqrt(d_k))")
    display(head0_attn.round(3))
    print("sentence vector (first 12 dims):", np.round(report["sentence_vector"][:12], 3))
    print()

sentence_1: автомобиль сломался
tokens: ['<s>', '▁автомобиль', '▁сло', 'ма', 'лся', '</s>']
Q shape: (1, 12, 6, 32)
K shape: (1, 12, 6, 32)
V shape: (1, 12, 6, 32)
head 0: Q


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
<s>,-1.997,-1.238,0.071,-0.789,-0.021,0.129,1.856,-0.317,-1.185,-0.786,0.098,-0.793,1.466,-0.972,-0.318,1.666,2.315,1.821,0.559,-0.951,0.390,0.324,-1.820,1.327,-0.120,1.231,0.594,0.397,-0.423,0.458,1.638,0.280
▁автомобиль,1.698,-1.190,1.084,0.358,0.517,0.986,-1.111,0.435,-0.272,-1.103,1.706,-1.063,-0.663,-0.421,0.234,-0.003,0.509,2.735,-1.020,-0.743,0.501,0.687,-1.476,-0.315,0.588,-1.180,-1.645,0.622,0.008,0.705,-0.803,0.872
▁сло,-0.589,-0.865,-0.527,0.490,-0.260,-1.123,0.127,1.595,0.487,-1.677,1.091,1.687,0.777,-0.740,0.702,1.857,0.467,0.984,-1.027,0.118,-0.728,0.784,-0.512,-0.809,0.521,0.278,-0.851,-0.822,0.556,0.851,0.201,0.976
ма,0.424,-0.581,1.523,0.691,-0.199,0.488,0.484,0.319,1.708,-0.934,0.892,-1.013,0.352,0.922,0.779,0.690,0.227,-0.207,-0.770,0.484,0.107,0.051,0.020,-1.158,-0.395,-0.724,0.209,1.422,-0.798,0.723,0.530,-0.114
лся,0.161,-0.215,1.621,0.853,1.640,-0.029,-1.063,0.249,-1.488,-0.023,-0.201,-0.386,0.469,-0.535,0.934,1.536,-0.657,0.755,-0.241,-0.698,-0.365,-0.450,-1.908,-1.387,0.375,-2.308,-1.159,-0.292,0.197,0.652,-0.572,0.516
</s>,-0.796,-0.514,0.860,0.508,0.729,0.690,0.694,-0.364,-1.129,-0.415,0.698,-0.238,0.995,-1.150,-0.005,1.123,0.601,1.033,0.296,-0.972,0.806,0.287,-1.067,1.165,-0.306,-0.287,0.090,1.441,-0.601,-0.845,0.862,0.343


head 0: K


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
<s>,-2.476,-1.734,-1.375,-1.283,-0.263,0.716,2.052,-0.028,-1.582,0.403,-0.267,-0.388,0.211,-0.443,-0.669,1.219,2.024,1.329,1.070,-1.239,-0.572,0.467,-3.167,0.109,-0.105,1.801,-0.307,0.518,-0.819,0.096,0.859,-0.425
▁автомобиль,0.674,-0.453,0.034,-0.834,-1.139,-0.601,-2.585,0.942,-0.671,-0.891,0.598,-0.055,0.426,0.461,0.659,1.577,-0.260,-0.877,0.024,-0.758,1.164,0.681,0.708,-1.254,-0.850,-0.738,-0.332,-0.150,-0.204,-0.566,0.530,0.078
▁сло,0.172,-0.285,-0.174,-0.431,1.485,0.374,-1.705,0.617,-0.664,-0.356,0.037,-1.880,-0.572,0.375,0.434,1.383,-0.585,0.205,0.721,-0.583,0.511,-0.908,-0.354,-0.838,-0.344,0.250,0.030,0.352,-0.410,-0.227,1.222,0.428
ма,-1.090,0.108,-3.325,-1.489,0.441,1.043,0.054,-0.355,-0.025,0.838,0.222,0.860,-2.357,-1.592,-0.213,0.286,-0.597,0.408,-1.589,-0.133,-0.114,-0.173,-0.572,0.120,0.908,1.468,-1.037,-2.135,0.358,-0.454,-1.644,1.087
лся,-0.343,0.109,-0.486,-0.305,-2.346,1.046,-0.993,2.502,0.078,-0.625,-0.411,0.229,-0.432,-0.356,1.617,-1.041,-0.385,-1.034,-0.421,1.024,0.298,-1.003,1.543,-0.284,1.790,-0.221,-0.032,-0.552,0.080,-1.222,1.119,1.724
</s>,-0.595,-0.330,-0.930,-2.294,0.159,0.673,0.600,0.520,-1.313,-0.697,-1.332,-0.271,0.710,-0.266,-0.591,0.737,1.589,-0.645,0.338,-0.643,-0.737,0.272,-3.234,0.112,0.699,2.585,-0.341,1.528,1.139,0.290,0.770,-0.934


head 0: V


...

head 0: V


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
<s>,0.045,0.052,0.017,0.024,-0.114,0.029,-0.006,-0.050,0.090,0.110,0.164,0.013,0.079,0.013,-0.029,-0.027,0.054,0.069,0.026,-0.067,0.115,0.011,-0.022,0.057,-0.121,-0.023,-0.085,-0.115,-0.004,0.028,-0.092,0.044
▁машина,0.903,0.349,-0.120,-0.197,0.061,0.344,-0.870,-0.271,-0.139,-0.142,-0.173,-0.057,1.046,-0.005,-0.295,-0.266,0.180,0.950,-0.041,-0.358,-0.088,-0.128,-0.553,0.699,-0.840,-0.377,0.700,-0.063,0.004,-0.652,0.272,0.568
▁вышла,-0.025,-0.037,-0.206,0.362,0.419,-0.369,-0.265,-0.507,-1.272,0.345,0.825,-0.488,0.476,0.585,0.243,0.308,-0.205,-0.135,0.089,0.183,0.058,-0.165,0.204,-0.395,0.317,-0.043,0.161,-0.077,-0.376,0.398,-0.514,-0.093
▁из,-0.062,0.671,-0.046,0.153,-0.461,0.192,0.095,-1.017,-0.204,-0.297,0.004,0.752,-0.364,-0.457,-0.486,0.600,-0.091,-0.311,0.404,0.534,0.044,-0.251,0.274,0.161,-0.398,-0.081,-0.037,0.746,-0.441,-0.044,-0.630,0.473
▁строя,0.631,0.446,-0.520,-0.115,0.091,0.428,-0.408,-0.514,0.378,1.405,0.632,-0.430,0.073,0.588,0.286,0.089,0.458,-0.522,-0.484,-0.312,0.661,-0.055,-0.989,-0.044,0.156,0.306,0.240,0.326,-1.022,-0.670,0.091,0.105
</s>,-0.002,0.101,-0.169,0.093,-0.089,-0.211,-0.171,0.025,0.099,0.365,0.159,0.399,-0.173,-0.033,-0.022,0.229,-0.132,-0.155,-0.098,-0.359,-0.170,-0.129,0.144,0.071,-0.053,-0.117,0.060,-0.236,0.073,-0.282,-0.096,0.082


head 0: attention weights A = softmax(QK^T / sqrt(d_k))


,<s>,▁машина,▁вышла,▁из,▁строя,</s>
<s>,0.899,0.002,0.001,0.002,0.003,0.093
▁машина,0.003,0.012,0.702,0.005,0.272,0.006
▁вышла,0.110,0.241,0.147,0.032,0.432,0.037
▁из,0.008,0.070,0.388,0.004,0.524,0.006
▁строя,0.266,0.272,0.138,0.057,0.142,0.125
</s>,0.562,0.036,0.037,0.023,0.205,0.137


sentence vector (first 12 dims): [ 0.043  0.128  0.31  -0.179  0.165 -0.115 -0.067  0.108 -0.068 -0.452
  0.145  0.399]



In [15]:
real_score = float(
    cosine_similarity(
        [real_reports[0]["sentence_vector"]],
        [real_reports[1]["sentence_vector"]],
    )[0, 0]
)

summary = pd.DataFrame(
    {
        "method": [
            "CountVectorizer",
            "TF-IDF",
            "Toy Transformer",
            "BERT multilingual",
        ],
        "cosine_similarity": [
            count_score,
            tfidf_score,
            toy_score,
            real_score,
        ],
    }
)

summary["cosine_similarity"] = summary["cosine_similarity"].round(4)
summary

,method,cosine_similarity
0,CountVectorizer,0.0000
1,TF-IDF,0.0000
2,Toy Transformer,0.9997
3,BERT multilingual,0.8538


## 4. Контрастные примеры

Три пары, которые хорошо показывают разницу между методами:

| Пара | TF-IDF | Transformer |
|------|--------|-------------|
| Одинаковые слова, разный смысл | ≈ 1.0 | << 1.0 |
| Разные слова, одинаковый смысл | ≈ 0.0 | >> 0.0 |
| Отрицание | ≈ высокий | ниже |

In [16]:
contrast_pairs = [
    {
        "label": "Одинаковые слова, разный смысл",
        "s1": "собака укусила мальчика",
        "s2": "мальчик укусил собаку",
    },
    {
        "label": "Разные слова, одинаковый смысл",
        "s1": "автомобиль сломался",
        "s2": "машина вышла из строя",
    },
    {
        "label": "Отрицание",
        "s1": "я люблю кошек",
        "s2": "я не люблю кошек",
    },
]

rows = []

for pair in contrast_pairs:
    s1, s2 = pair["s1"], pair["s2"]

    # TF-IDF
    tfidf_mat = TfidfVectorizer().fit_transform([s1, s2]).toarray()
    tfidf_sim = float(cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:2])[0, 0])

    # Transformer
    r1 = real_transformer_report(s1)
    r2 = real_transformer_report(s2)
    bert_sim = float(cosine_similarity([r1["sentence_vector"]], [r2["sentence_vector"]])[0, 0])

    rows.append({
        "label": pair["label"],
        "sentence_1": s1,
        "sentence_2": s2,
        "tfidf_cosine": round(tfidf_sim, 4),
        "transformer_cosine": round(bert_sim, 4),
        "delta": round(tfidf_sim - bert_sim, 4),
    })

contrast_df = pd.DataFrame(rows)
contrast_df

,label,sentence_1,sentence_2,tfidf_cosine,transformer_cosine,delta
0,"Одинаковые слова, разный смысл",собака укусила мальчика,мальчик укусил собаку,0.0000,0.9826,-0.9826
1,"Разные слова, одинаковый смысл",автомобиль сломался,машина вышла из строя,0.0000,0.8538,-0.8538
2,Отрицание,я люблю кошек,я не люблю кошек,0.7093,0.5672,0.1420
